In [8]:
import pandas as pd
from pathlib import Path

csv_path = Path("data/final/master_dataset.csv")
if not csv_path.exists():
    csv_path = Path("../data/final/master_dataset.csv")

print("Đang đọc:", csv_path.resolve())
df = pd.read_csv(csv_path)


Đang đọc: T:\nhom_ptdl\data-analysis-ai-nhom11\data\final\master_dataset.csv


In [9]:
print("Số dòng, số cột:", df.shape)
print("\nTên các cột:")
print(df.columns.tolist())
print("\nKiểu dữ liệu:")
print(df.dtypes)
print("\nGiá trị thiếu theo cột:")
print(df.isna().sum())

print("\nTransaction_ID duy nhất:", df["Transaction_ID"].is_unique)
print("Transaction_ID bị thiếu:", df["Transaction_ID"].isna().sum())

Số dòng, số cột: (10000, 39)

Tên các cột:
['Transaction_ID', 'Transaction_Date', 'Customer_ID', 'Gender', 'Age', 'City', 'Product_ID', 'Product_Name', 'Category', 'Brand', 'Quantity', 'Unit_Price_VND', 'Revenue', 'Source_Currency', 'Rating', 'Customer_Review', 'Original_Price', 'Discount_Price', 'Original_Price_VND', 'Discount_Price_VND', 'avg_rating', 'reviews_count', 'rating_average', 'review_count', 'quantity_sold', 'availability', 'is_for_sale', 'description', 'product_type', 'category_2', 'category_3', 'manufacturer', 'current_seller', 'fulfillment_type', 'date_created', 'number_of_images', 'favourite_count', 'has_video', 'Source']

Kiểu dữ liệu:
Transaction_ID         object
Transaction_Date       object
Customer_ID            object
Gender                 object
Age                     int64
City                   object
Product_ID              int64
Product_Name           object
Category               object
Brand                  object
Quantity                int64
Unit_Pric

In [10]:
schema = pd.DataFrame({
    "Cot": df.columns,
    "Kieu_Pandas": df.dtypes.astype(str).values,
    "So_Gia_Tri_Thieu": df.isna().sum().values
})

print(schema.to_string(index=False))

               Cot Kieu_Pandas  So_Gia_Tri_Thieu
    Transaction_ID      object                 0
  Transaction_Date      object                 0
       Customer_ID      object                 0
            Gender      object                 0
               Age       int64                 0
              City      object                 0
        Product_ID       int64                 0
      Product_Name      object                 0
          Category      object                 0
             Brand      object                 0
          Quantity       int64                 0
    Unit_Price_VND       int64                 0
           Revenue       int64                 0
   Source_Currency      object                 0
            Rating       int64                 0
   Customer_Review      object                 0
    Original_Price     float64                 0
    Discount_Price     float64                 0
Original_Price_VND     float64                 0
Discount_Price_VND  

In [11]:
print(schema.iloc[20:36].to_string(index=False))

             Cot Kieu_Pandas  So_Gia_Tri_Thieu
      avg_rating      object              8178
   reviews_count      object              8178
  rating_average     float64              1822
    review_count     float64              1822
   quantity_sold     float64              1822
    availability      object              8178
     is_for_sale      object              8178
     description      object                 0
    product_type      object              8178
      category_2      object              8178
      category_3      object              8178
    manufacturer      object              8178
  current_seller      object              1822
fulfillment_type      object              1822
    date_created     float64              1822
number_of_images     float64              1822


In [12]:
# Kiểm tra có kết nói với sql không?
import pyodbc

print("Các ODBC driver trên máy:")
print(pyodbc.drivers())

Các ODBC driver trên máy:
['SQL Server', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 17 for SQL Server', 'EA Firebird ODBC driver', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']


In [13]:
from getpass import getpass
import pyodbc

server = r"DESKTOP-46JDDLG\SQLEXPRESS"  # Đối chiếu với Server name trong SSMS
password = getpass("Nhập mật khẩu tài khoản sa: ")

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={server};"
    "DATABASE=RetailAnalytics;"
    "UID=sa;"
    f"PWD={password};",
    timeout=10
)

count = conn.cursor().execute(
    "SELECT COUNT(*) FROM dbo.MasterTransactions"
).fetchone()[0]

print("Kết nối thành công. Số dòng hiện tại:", count)
conn.close()

Kết nối thành công. Số dòng hiện tại: 0


In [ ]:

%run ../src/database/load_sqlserver.py
# nạp ở đây xong  qua sql kiểm tra

Số giao dịch trong CSV: 10000
Số giao dịch vừa nạp: 10000
Tổng số giao dịch trong SQL Server: 10000


In [16]:
# Đọc từ sql về python

import pandas as pd
import pyodbc
from getpass import getpass

server = r"DESKTOP-46JDDLG\SQLEXPRESS"
password = getpass("Nhập mật khẩu sa: ")

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={server};"
    "DATABASE=RetailAnalytics;"
    "UID=sa;"
    f"PWD={password};"
)

cursor = conn.cursor()
cursor.execute("SELECT * FROM dbo.MasterTransactions")

columns = [column[0] for column in cursor.description]
rows = [tuple(row) for row in cursor.fetchall()]
df_sql = pd.DataFrame(rows, columns=columns)
conn.close()

print("Kích thước dữ liệu đọc từ SQL Server:", df_sql.shape)
display(df_sql.head())

Kích thước dữ liệu đọc từ SQL Server: (10000, 39)


,Transaction_ID,Transaction_Date,Customer_ID,Gender,Age,City,Product_ID,Product_Name,Category,Brand,...,category_2,category_3,manufacturer,current_seller,fulfillment_type,date_created,number_of_images,favourite_count,has_video,Source
0,T000001,2026-05-27 07:53:23,C01049,Male,31,TP. Hồ Chí Minh,173798774,Hũ chiết kem dưỡng cao cấp nút nhân Rabbit Lab...,Túi chống sốc,OEM,...,None,None,None,Rabbit Lab,dropship,525.0,3.0000,0.0000,False,Tiki
1,T000002,2025-11-03 05:40:50,C00099,Female,47,Vũng Tàu,24659536,Bộ Chiết Mỹ Phẩm Du Lịch,Thời Trang,OEM,...,None,None,None,Hitech07,dropship,1504.0,7.0000,0.0000,False,Tiki
2,T000003,2025-09-30 17:07:01,C00503,Male,23,Huế,16683725,Balo túi du lịch đa năng Ozuko Z9060 (Đen),Thời Trang,OZUKO,...,None,None,None,Tỷ Lợi,dropship,1572.0,3.0000,0.0000,False,Tiki
3,T000004,2026-07-05 23:45:05,C00810,Male,59,Hà Nội,570490,Túi Kéo Du Lịch Macat Innova 4 - Đỏ,Thời Trang,Macat,...,None,None,None,MACAT OFFICIAL,dropship,0.0,5.0000,0.0000,False,Tiki
4,T000005,2026-01-05 19:27:32,C00241,Female,59,TP. Hồ Chí Minh,325087836,Nicola Spring Coir Door Mat - 60 x 40cm - Blac...,Marketplace,Nicola Spring,...,Home Accessories,Rugs & Mats,\N,None,None,None,None,None,None,Tesco


In [ ]:
# Kết nối mới mongo alast
%pip install pymongo

   ---------------------------------------- 0.0/822.0 kB ? eta -:--:--
   ------------ --------------------------- 262.1/822.0 kB ? eta -:--:--
   ---------------------------------------- 822.0/822.0 kB 6.6 MB/s  0:00:00

   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   -------------------- ------------------- 1/2 [pymongo]
   -------------------- ------------------- 1/2 [pymongo]
   -------------------- ------------------- 1/2 [pymongo]
   -------------------- ------------------- 1/2 [pymongo]
   -------------------- ------------------- 1/2 [pymongo]
   -------------------- ------------------- 1/2 [pymongo]
   -------------------- ------------------- 1/2 [pymongo]
   -------------------- ------------------- 1/2 [pymongo]
   -------------------- ------------------- 1/2 [pymongo]
   -------------------- ------------------- 1/2 [pymongo]
   ---------------

In [ ]:
from getpass import getpass
from urllib.parse import quote_plus
from pymongo import MongoClient

uri_mau = input("Dán chuỗi mongodb+srv (giữ nguyên <db_password>): ").strip()
mat_khau = getpass("Nhập mật khẩu database user: ")

uri = uri_mau.replace("<db_password>", quote_plus(mat_khau))
client = MongoClient(uri, serverSelectionTimeoutMS=15000)

print(client.admin.command("ping"))
client.close()



# mongodb+srv://phanchithanh1lavn_db_user:<db_password>@cluster0.bgqtb4z.mongodb.net/?appName=Cluster0

# mk: @Aa00****

{'ok': 1}


In [ ]:
# Database name: retail_analytics
# Collection name: transactions

In [19]:
# Nạp dữ liệu bằng python
%run ../src/database/load_mongodb.py

Đã xử lý 500/10000 giao dịch
Đã xử lý 1000/10000 giao dịch
Đã xử lý 1500/10000 giao dịch
Đã xử lý 2000/10000 giao dịch
Đã xử lý 2500/10000 giao dịch
Đã xử lý 3000/10000 giao dịch
Đã xử lý 3500/10000 giao dịch
Đã xử lý 4000/10000 giao dịch
Đã xử lý 4500/10000 giao dịch
Đã xử lý 5000/10000 giao dịch
Đã xử lý 5500/10000 giao dịch
Đã xử lý 6000/10000 giao dịch
Đã xử lý 6500/10000 giao dịch
Đã xử lý 7000/10000 giao dịch
Đã xử lý 7500/10000 giao dịch
Đã xử lý 8000/10000 giao dịch
Đã xử lý 8500/10000 giao dịch
Đã xử lý 9000/10000 giao dịch
Đã xử lý 9500/10000 giao dịch
Đã xử lý 10000/10000 giao dịch
Số dòng CSV: 10000
Số documents MongoDB: 10000


In [20]:
# đọc dữ liệu từ mongodb vè python
import pandas as pd
from getpass import getpass
from urllib.parse import quote_plus
from pymongo import MongoClient

uri_mau = input("Dán chuỗi mongodb+srv còn <db_password>: ").strip()
mat_khau = getpass("Nhập mật khẩu database user: ")
uri = uri_mau.replace("<db_password>", quote_plus(mat_khau))

client = MongoClient(uri, serverSelectionTimeoutMS=15000)
try:
    documents = list(
        client["retail_analytics"]["transactions"].find({}, {"_id": 0})
    )
    df_mongo = pd.DataFrame(documents)
    print("Kích thước dữ liệu từ MongoDB:", df_mongo.shape)
    display(df_mongo.head())
finally:
    client.close()

Kích thước dữ liệu từ MongoDB: (10000, 39)


,Transaction_ID,Transaction_Date,Customer_ID,Gender,Age,City,Product_ID,Product_Name,Category,Brand,...,category_2,category_3,manufacturer,current_seller,fulfillment_type,date_created,number_of_images,favourite_count,has_video,Source
0,T000001,2026-05-27 07:53:23,C01049,Male,31,TP. Hồ Chí Minh,173798774,Hũ chiết kem dưỡng cao cấp nút nhân Rabbit Lab...,Túi chống sốc,OEM,...,None,None,None,Rabbit Lab,dropship,525.0,3.0,0.0,False,Tiki
1,T000002,2025-11-03 05:40:50,C00099,Female,47,Vũng Tàu,24659536,Bộ Chiết Mỹ Phẩm Du Lịch,Thời Trang,OEM,...,None,None,None,Hitech07,dropship,1504.0,7.0,0.0,False,Tiki
2,T000003,2025-09-30 17:07:01,C00503,Male,23,Huế,16683725,Balo túi du lịch đa năng Ozuko Z9060 (Đen),Thời Trang,OZUKO,...,None,None,None,Tỷ Lợi,dropship,1572.0,3.0,0.0,False,Tiki
3,T000004,2026-07-05 23:45:05,C00810,Male,59,Hà Nội,570490,Túi Kéo Du Lịch Macat Innova 4 - Đỏ,Thời Trang,Macat,...,None,None,None,MACAT OFFICIAL,dropship,0.0,5.0,0.0,False,Tiki
4,T000005,2026-01-05 19:27:32,C00241,Female,59,TP. Hồ Chí Minh,325087836,Nicola Spring Coir Door Mat - 60 x 40cm - Blac...,Marketplace,Nicola Spring,...,Home Accessories,Rugs & Mats,\N,None,None,NaN,NaN,NaN,None,Tesco


## Thiết kế lưu trữ và lý do lựa chọn

# Nhóm sử dụng hai công nghệ: SQL Server và MongoDB Atlas.

# - **SQL Server:** Lưu 10.000 giao dịch trong bảng `RetailAnalytics.dbo.MasterTransactions` với 39 cột. `Transaction_ID` là khóa chính,
#  giúp mỗi giao dịch chỉ xuất hiện một lần. Cấu trúc bảng phù hợp để truy vấn và tổng hợp doanh thu, số lượng bán, đánh giá bằng SQL.
# - **MongoDB Atlas:** Lưu cùng 10.000 giao dịch trong `retail_analytics.transactions`. 
# Mỗi giao dịch là một document; `_id` lấy từ `Transaction_ID`. Cách này thuận tiện cho dữ liệu sản phẩm đến từ Tesco và Tiki, vì một số thuộc tính chỉ có ở một nguồn.
# - **Giá trị thiếu:** Khi nạp dữ liệu, các giá trị thiếu được giữ dưới dạng `NULL` trong SQL Server và `null` trong MongoDB; việc xử lý thiếu dữ liệu thuộc Bước 5.
# - **Kiểm chứng:** Đã truy xuất dữ liệu từ cả hai hệ thống về pandas. Mỗi hệ thống trả về 10.000 giao dịch với 39 thuộc tính để phục vụ EDA.